In [ ]:
import torch
import torch.nn as nn
import torch_pruning as tp

model = nn.TransformerEncoderLayer(d_model=64, nhead=4)
example_input = torch.randn(10, 1, 64)  # [seq_len, batch_size, dim]

# Capture activations
captured_activations = {}

def save_activation(module, input, output):
    captured_activations['linear2_input'] = input[0].detach()
    captured_activations['linear2_output'] = output.detach()

# Register hook
hook_handle = model.linear2.register_forward_hook(save_activation)

# Run forward pass
with torch.no_grad():
    output = model(example_input)

# Remove hook
hook_handle.remove()

# Compute importance from saved output
class ActivationImportance(tp.importance.Importance):
    def __call__(self, layer, inputs, outputs):
        activation = outputs[0]
        return activation.abs().mean(dim=(0, 1))  # shape [output_dim]

importance = ActivationImportance()
scores = importance(model.linear2,
                    (captured_activations['linear2_input'],),
                    (captured_activations['linear2_output'],))

# Select least important neurons
_, prune_idxs = torch.topk(scores, k=16, largest=False)

# Build pruning graph
DG = tp.DependencyGraph()
DG.build_dependency(model, example_inputs=(example_input,))

# Prune
group = DG.get_pruning_group(
    module=model.linear2,
    pruning_fn=tp.ops.,  # or tp.prune_linear_in depending on direction
    idxs=prune_idxs.tolist()
)
group.prune()

print(model)


AttributeError: module 'torch_pruning.ops' has no attribute 'prune_linear_out'